<a href="https://colab.research.google.com/github/naishvi1705/naishvi_shah_project/blob/main/sephora_makeup_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/sephora_project/product_info.csv')
df.shape

(8494, 27)

In [5]:
# Keep only makeup
makeup = df[df['primary_category'] == 'Makeup']

# Remove non-formulated products
exclude = ['Brushes & Applicators', 'Accessories', 'Value & Gift Sets', 'Mini Size']
makeup = makeup[~makeup['secondary_category'].isin(exclude)].copy()

print(makeup.shape)
makeup['secondary_category'].value_counts()

(2018, 27)


,count
secondary_category,
Eye,711
Face,659
Lip,411
Cheek,165
Nail,52
Makeup Palettes,20


In [6]:
import ast

makeup['highlights_list'] = makeup['highlights'].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else []
)

makeup[['product_name', 'highlights', 'highlights_list']].head()

,product_name,highlights,highlights_list
65,Rose Lip Conditioner,"['Hydrating', 'Good for: Dryness']","[Hydrating, Good for: Dryness]"
189,Lip Treatment Oil,"['Vegan', 'High Shine Finish', 'Clean at Sepho...","[Vegan, High Shine Finish, Clean at Sephora, H..."
190,Skin-Enhancing Tinted Moisturizer,"['Vegan', 'Natural Finish', 'Clean at Sephora'...","[Vegan, Natural Finish, Clean at Sephora, Medi..."
191,Lash-Amplifying Volumizing & Lengthening Mascara,"['Vegan', 'allure 2022 Best of Beauty Award Wi...","[Vegan, allure 2022 Best of Beauty Award Winne..."
192,Skin Melt Talc-Free Loose Setting Powder,"['Vegan', 'Loose Powder Formula', 'Clean at Se...","[Vegan, Loose Powder Formula, Clean at Sephora..."


In [7]:
claim_groups = {
    'Ethical/clean': ['Vegan', 'Cruelty-Free', 'Clean at Sephora', 'Without Parabens',
                      'Without Silicones', 'Fragrance Free'],
    'Performance':   ['Long-wearing', 'Waterproof', 'Hydrating'],
    'Finish':        ['Matte Finish', 'Natural Finish', 'Radiant Finish', 'Satin Finish',
                      'Shimmer Finish', 'High Shine Finish', 'Metallic Finish']
}

for group, claims in claim_groups.items():
    for claim in claims:
        makeup['claim_' + claim] = makeup['highlights_list'].apply(lambda tags: claim in tags)

# Flag whether a product has any tags at all (for fair comparisons later)
makeup['has_highlights'] = makeup['highlights_list'].str.len() > 0

# Count how many products carry each claim
makeup.filter(like='claim_').sum().sort_values(ascending=False)

,0
claim_Vegan,756
claim_Long-wearing,621
claim_Cruelty-Free,448
claim_Without Parabens,410
claim_Matte Finish,361
claim_Hydrating,350
claim_Clean at Sephora,264
claim_Natural Finish,262
claim_Radiant Finish,176
claim_Waterproof,152


In [8]:
makeup['price_tier'] = pd.cut(makeup['price_usd'],
                              bins=[0, 20, 35, 1000],
                              labels=['Under $20', '$20–35', 'Over $35'])

makeup['price_tier'].value_counts()

,count
price_tier,
$20–35,974
Over $35,678
Under $20,366


In [11]:
import warnings
warnings.filterwarnings('ignore', category=SyntaxWarning)

makeup['ingredients_text'] = makeup['ingredients'].apply(
    lambda x: ' '.join(ast.literal_eval(x)).lower() if pd.notna(x) else ''
)
makeup['has_ingredients'] = makeup['ingredients_text'] != ''

makeup['has_ingredients'].sum()

np.int64(1923)

In [12]:
silicones = r'methicone|siloxane|silsesquioxane|silicone|siloxysilicate'

microplastic_assoc = (r'nylon-\d+|polyethylene(?! glycol)|polypropylene(?! glycol)|'
                      r'polymethyl methacrylate|methyl methacrylate crosspolymer|'
                      r'polystyrene|ptfe|polytetrafluoroethylene|polyurethane|'
                      r'polybutylene terephthalate')

makeup['contains_silicone'] = makeup['ingredients_text'].str.contains(silicones)
makeup['contains_microplastic_assoc'] = makeup['ingredients_text'].str.contains(microplastic_assoc)

# Percentages among products that list ingredients
with_ing = makeup[makeup['has_ingredients']]
print('Silicone:', round(with_ing['contains_silicone'].mean() * 100, 1), '%')
print('Microplastic-associated:', round(with_ing['contains_microplastic_assoc'].mean() * 100, 1), '%')

Silicone: 56.4 %
Microplastic-associated: 36.2 %


In [13]:
import re

# Products that claim "Without Silicones" AND list ingredients
silicone_free_claims = makeup[makeup['claim_Without Silicones'] & makeup['has_ingredients']]
print('Products claiming silicone-free (with INCI):', len(silicone_free_claims))

# Which of those still contain a silicone?
contradictions = silicone_free_claims[silicone_free_claims['contains_silicone']]
print('Of those, containing a silicone:', len(contradictions))

# Show the product and the exact silicone ingredient(s) found
contradictions = contradictions.assign(
    silicones_found = contradictions['ingredients_text'].apply(
        lambda t: sorted(set(m.strip() for m in re.findall(r'[^,:]*(?:' + silicones + r')[^,]*', t)))
    )
)
contradictions[['brand_name', 'product_name', 'tertiary_category', 'silicones_found']]

Products claiming silicone-free (with INCI): 62
Of those, containing a silicone: 3


,brand_name,product_name,tertiary_category,silicones_found
441,bareMinerals,Gen Nude Powder Blush,Blush,"[dimethicone, hydrogen dimethicone, vinyl dime..."
4530,LAWLESS,Hold Up Soft Set Creamy Eyebrow Wax,Eyebrow,[simethicone]
7761,TOM FORD,Ultra Shine Lip Color,Lipstick,[phenyl trimethicone]


In [14]:
# Flag the 3 contradictions in the main table
makeup['silicone_claim_mismatch'] = makeup['claim_Without Silicones'] & makeup['contains_silicone']

keep = (['product_id', 'brand_name', 'product_name',
         'secondary_category', 'tertiary_category',
         'price_usd', 'price_tier', 'rating', 'reviews', 'loves_count',
         'has_highlights', 'has_ingredients',
         'contains_silicone', 'contains_microplastic_assoc', 'silicone_claim_mismatch']
        + [c for c in makeup.columns if c.startswith('claim_')])

export = makeup[keep]
export.to_csv('/content/drive/MyDrive/sephora_project/makeup_clean.csv', index=False)

print(export.shape)

(2018, 31)
